<a href="https://colab.research.google.com/github/mshmatsuda/deep-learning-from-scratch/blob/main/03_6_03_%E3%83%90%E3%83%83%E3%83%81%E5%87%A6%E7%90%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## バッチ処理

まず重みの形状（形）」を最初に出力させたのは、「1枚ずつ処理していた計算を、何枚もまとめて処理（バッチ処理）するときの作戦を立てる

## ディープラーニングで一番の難所と言われる「行列の形（シェイプ）」の話

------------------------------
## 1. 画面に出力された「形状（Shape）」の意味
プログラムを実行して重みの形（.shape）を出力すると、以下のような数字が表示されます。

* 入力（画像1枚）: (784,) ➔ 28×28マスのピクセルが1列に並んだもの
* 重み W1: (784, 50) ➔ 第1層（隠れ層）の重み
* 重み W2: (50, 100) ➔ 第2層（隠れ層）の重み
* 重み W3: (100, 10) ➔ 出力層（0〜9の10個の数字）の重み

これは、ニューラルネットワークの部屋（層）の「入り口の数」と「出口の数」を表しています。
例えば、W1 の (784, 50) は、「784個のデータが入ってきたら、計算して50個にして次の部屋へ送り出す」という意味の巨大な数字のテーブルです。

------------------------------
## 2. 「1枚ずつ」のときの計算の形（復習）
これまでやってきた「1枚ずつループで回す」ときの計算は、NumPyの行列計算（np.dot）で以下のような「形の変化」をしていました。

* x の形： 784
* W1 の形： 748 × 50
* 計算結果（np.dot(x, W1)）： 50

行列の掛け算は、「左側の後ろの数字」と「右側の前の数字」が一致していないと計算できないという絶対のルールがあります（今回は 784 と 784 で一致しています）。

------------------------------
## 3. なぜ「重みの形状」を確認する必要があるのか？
今からやりたい「バッチ処理」とは、画像を1枚ずつではなく、例えば「100枚の束（バッチ）」にして一気にネットワークに放り込む技です。<br>
画像を100枚（もともと10000枚ある）まとめると、入力データの形は (784,) から (100, 784) という、100行×784列の巨大な表に変わります。<br><br>
ここで、本が言いたかった一番大切なポイントです。<br>
「入力を100枚に増やしたとき、今の重み（W1, W2, W3）の形のままで、エラーを起こさずに行列の掛け算ができるか？」を確認したかったのです。<br><br>
実際に形を並べてみましょう。

* 100枚の入力の形： 100 × 784
* 重み W1 の形： 784 × 50

左の後ろ（784）と、右の前（784）がピッタリ一致しています。<br>
行列のルール通り、これらはそのまま掛け算が可能で、計算結果は 100 × 50（100枚分の、50個のデータ）という形になって出てきます。<br><br>

## 💡 結論として何がわかったのか？
重みの形状を確認したことで、「重みの形（ネットワークの構造）は一切変えなくても、入力するデータの側を 100枚の束（100, 784）にするだけで、Pythonは自動的に100枚分を同時に一瞬で計算してくれる！」ということが証明されたわけです。



In [2]:
# ①必要なデータをダウンロード

domain = "https://github.com"
repo_path = "/oreilly-japan/deep-learning-from-scratch.git"
full_url = domain + repo_path

!git clone {full_url}

Cloning into 'deep-learning-from-scratch'...
remote: Enumerating objects: 573, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 573 (delta 3), reused 0 (delta 0), pack-reused 558 (from 2)
Receiving objects: 100% (573/573), 10.61 MiB | 29.28 MiB/s, done.
Resolving deltas: 100% (290/290), done.


In [9]:
# ②作業フォルダを移動

%cd /content/deep-learning-from-scratch

/content/deep-learning-from-scratch


In [10]:
import sys,os
sys.path.append(os.pardir)
import numpy as np

import pickle

from dataset.mnist import load_mnist
from common.functions import sigmoid, softmax


In [12]:
# 1. データの読み込み（今回は正解ラベル t も使うので両方受け取ります）
def get_data():
    (_, _), (x_test, t_test) = load_mnist(normalize=True, flatten=True, one_hot_label=False)
    return x_test, t_test

# 2. 学習済みモデル（重み）の読み込み
def init_network():
    with open("ch03/sample_weight.pkl", 'rb') as f:
        network = pickle.load(f)
    return network

# 3. 推論（予測）関数
def predict(network, x):
    W1, W2, W3 = network['W1'], network['W2'], network['W3']
    b1, b2, b3 = network['b1'], network['b2'], network['b3']

    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)
    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)
    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y

# --- ここからがバッチ処理のメイン ---
x, t = get_data()
network = init_network()

batch_size = 100 # 100枚ずつの束（バッチ）にする
accuracy_cnt = 0

# 1万枚を「100マス飛ばし」でループする
for i in range(0, len(x), batch_size):
    x_batch = x[i:i+batch_size] # 100枚の画像をガサッと切り出す
    y_batch = predict(network, x_batch) # 100枚一気にAIに予測させる

    p = np.argmax(y_batch, axis=1) # 100枚それぞれの一番確率が高い数字の番号を取得
    accuracy_cnt += np.sum(p == t[i:i+batch_size]) # 100枚分の正解数を一気に合計して足す

print("Accuracy:" + str(float(accuracy_cnt) / len(x)))
# 出力結果: Accuracy:0.9352


Accuracy:0.9352


## 解説

いよいよバッチ処理の最重要パート。<br>
ここが理解できると、ディープラーニングのコードが劇的に読めるようになります。<br>
1枚ずつループしていたときとの違いを意識しながら、1回目、2回目とループが進む様子を見ていきます。


------------------------------
## 1. for i in range(0, len(x), batch_size): の動き
この range(0, 10000, 100) は、「0から10000の手前まで、100ずつ飛ばした数字の列」を作ります。<br>
そのため、変数 i には、ループが回るたびに以下のように数字が入ります。

* 1回目： i = 0
* 2回目： i = 100
* 3回目： i = 200
* …… 最後の100回目： i = 9900

つまり、1万回まわしていたループが、たったの100回（100回 × 100枚 ＝ 1万枚）に激減します。

------------------------------
## 2. ループの中身のリアルな実況中継
この特殊な for ループがどう動くか、中身を見てみましょう。
## 🔄【1回目：i = 0 のとき】x_batch = x[i:i + batch_size]

x_batch = x[0:100]


* 解説: Pythonの「スライス機能（[スタート : ゴール]）」を使い、1万枚ある画像データの中から 「0番目から99番目までの100枚」 をガサッと一気に切り出します。
* データの形: x_batch の形は、(100, 784)（100枚分の画像）という巨大な表になります。

y_batch = predict(network, x_batch)


* 解説: 切り出した100枚の画像（x_batch）を、そのまま予測関数に放り込みます。
* 何が起きる？: NumPyが内部で自動的に「100枚同時の行列計算（np.dot）」を一瞬で行います。
* 結果データ: y_batch の中には、100枚分の予測結果（それぞれ0〜9の確率が入ったデータ） がまとめて返ってきます。

------------------------------
## 🔄【2回目：i = 100 のとき】

x_batch = x[100:200]


* 解説: 自動的に次のターゲットである 「100番目から199番目までの100枚」 をガサッと切り出します。

y_batch = predict(network, x_batch)


* 解説: 次の100枚の画像をまとめて一気にAIに予測させ、次の100枚分の予測結果（y_batch）を手に入れます。

------------------------------
## 💡 1枚ずつのときとの最大の違い
1枚ずつのときは、「画像を1枚取り出す ➔ 予測する」を1万回細かく繰り返していました。<br>
バッチ処理では、「100枚の束を作る ➔ 束のまま一気に予測する」 を100回やるだけです。<br>
Pythonは細かいループを何度も回すのが苦手ですが、NumPyを使った大きな行列計算は超得意です。そのため、この書き方に変えるだけで、Colab上での計算スピードが文字通り「一瞬」に跳ね上がります。<br>


## 解説２

「100枚分の予測（y_batch）から、どうやって100枚分の答え合わせをまとめて行うのか（np.argmax の axis=1 や np.sum の部分）」の解説

この100枚一気の答え合わせパートは、バッチ処理の中で最も高度で、NumPyの強力な機能が詰め込まれている場所です。<br>
コードの以下の2行を、図解するように丁寧に紐解いていきます。



p = np.argmax(y_batch, axis=1)<br>
accuracy_cnt += np.sum(p == t[i:i+batch_size])

------------------------------
## 1. p = np.argmax(y_batch, axis=1) の動き
100枚一気に予測させた結果の y_batch は、「100行 × 10列」の巨大な表（2次元配列）になっています。

* 横の1行：画像1枚に対する、0から9の数字それぞれの確率（10個）
* 縦の100行：それが100枚分ズラリと並んでいる

ここで np.argmax(..., axis=1) を使うと、「横方向（1行ずつ）に見て、それぞれ一番確率が高い場所の番号（0〜9）を見つけてね」という命令になります。
## イメージ図：

          [ 0の確率,  1の確率,  2の確率, ... ,  9の確率 ]
1枚目：   [  0.01  ,   0.02 ,   0.90 , ... ,   0.01 ]  ➔ 一番高いのは「2」<br>
2枚目：   [  0.05  ,   0.85 ,   0.05 , ... ,   0.01 ]  ➔ 一番高いのは「1」<br>
...<br>
100枚目： [  0.70  ,   0.10 ,   0.05 , ... ,   0.02 ]  ➔ 一番高いのは「0」<br>

これを axis=1 で一括処理すると、100枚分の予測結果が詰まった要素数100個の1次元配列（リストのようなもの）が一瞬で出来上がります。

* 計算後の p の中身： [2, 1, ..., 0] （100枚分のAIの予想）

------------------------------
## 2. p == t[i:i+batch_size] で一斉に答え合わせ
次に、AIの予想（p）と、本当の正解（t から切り出した100枚分）をガチャンと重ね合わせて答え合わせをします。
Pythonの通常の関係演算子（==）をNumPy配列に対して使うと、「それぞれの要素を上から順番に比較して、合っていれば True、間違っていれば False」 を敷き詰めた、新しい100個の配列を作ってくれます。
## イメージ図：

* AIの予想 p： [2, 1, ..., 0]
* 本当の正解 t： [2, 3, ..., 0]
* 比較した結果（p == t）： [True, False, ..., True] （100個並ぶ）

------------------------------
## 3. np.sum(...) で正解数を一気に数え上げる
最後に、この [True, False, ..., True] という結果の合計を np.sum() で計算します。
ここでNumPyの面白いルールがあります。NumPyは足し算をするとき、True を 1、False を 0 として扱います。
そのため、この配列をすべて合計（np.sum）すると、「True（正解）の数だけがカウントされて、そのバッチでの正解数（例：95）」が数字として飛び出してきます。

accuracy_cnt += np.sum(p == t[i:i+batch_size])# 100枚の中の正解数（例: 95）が、これまでの合計カウンターにプラスされる！

------------------------------
## 💡 振り返ってみると
1枚ずつ処理していたときは、「if p == t[i]: accuracy_cnt += 1」という条件分岐を1万回判定していました。<br>
バッチ処理では、「100枚分を重ね合わせて、True/Falseの表を作り、Trueの数を一気に合計する」という方法に変わっています。<br>
if 文による細かい判定を一切使わず、すべてをNumPyの「配列の計算」に置き換えたことで、コンピューターが迷うことなく超ハイスピードで処理できるようになったわけです。<br>
